# Traffic Demand Prediction System

This notebook implements a high-performance machine learning system to predict traffic demand (`demand`) on Day 49 using historical data from Day 48 and early morning Day 49 (0:00 to 2:00).

## Approach Overview
1. **Geospatial Processing**: Decoded 6-character geohashes into latitude and longitude coordinates.
2. **Temporal Features**: Parsed raw timestamps into hours, minutes, minutes from midnight, and cyclical features (sine/cosine representation).
3. **Hierarchical Geospatial Aggregations**: Created geohash prefix features (length 4 and 5) and aggregated average historical demands inside these prefix neighborhoods. This helps the models generalize well to new or sparsely recorded locations.
4. **Lag and Rolling Features**: Extracted demand patterns from Day 48 at the corresponding times $t$, $t \pm 15$, $t \pm 30$, and $t \pm 60$ minutes, along with rolling averages and overall daily location metrics.
5. **Same-Day morning scaling**: Extracted early morning same-day demand statistics (from 0:00 to 2:00) to adjust predictions for the current day's overall scale.
6. **Ensembled Modeling**: Trained an ensemble of LightGBM, XGBoost, and CatBoost regressor models using a 5-fold cross-validation scheme on Day 49.

## Step 1: Imports and Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

## Step 2: Preprocessing and Utility Functions

In [ ]:
def decode_geohash(geohash):
    """
    Decodes a geohash string into latitude and longitude.
    """
    if not isinstance(geohash, str) or len(geohash) == 0:
        return np.nan, np.nan
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    
    for char in geohash:
        if char not in base32:
            return np.nan, np.nan
        val = base32.index(char)
        for i in range(4, -1, -1):
            bit = (val >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2.0
                if bit == 1:
                    lon_interval = (mid, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2.0
                if bit == 1:
                    lat_interval = (mid, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], mid)
            is_even = not is_even
            
    lat = (lat_interval[0] + lat_interval[1]) / 2.0
    lon = (lon_interval[0] + lon_interval[1]) / 2.0
    return lat, lon

def parse_time_features(df):
    """
    Extracts time features from the timestamp column.
    """
    df = df.copy()
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclic features
    df['sin_time'] = np.sin(2 * np.pi * df['time_minutes'] / 1440.0)
    df['cos_time'] = np.cos(2 * np.pi * df['time_minutes'] / 1440.0)
    
    return df

def impute_missing_values(df, train_df=None):
    """
    Imputes missing values for RoadType, Weather, and Temperature.
    """
    df = df.copy()
    ref_df = train_df if train_df is not None else df
    
    # Modes
    road_mode = ref_df['RoadType'].mode()[0] if not ref_df['RoadType'].mode().empty else 'Residential'
    weather_mode = ref_df['Weather'].mode()[0] if not ref_df['Weather'].mode().empty else 'Sunny'
    
    df['RoadType'] = df['RoadType'].fillna(road_mode)
    df['Weather'] = df['Weather'].fillna(weather_mode)
    
    # Temperature mapping
    temp_map = ref_df.groupby(['hour', 'Weather'])['Temperature'].mean().to_dict()
    overall_mean = ref_df['Temperature'].mean()
    if np.isnan(overall_mean):
        overall_mean = 20.0
        
    def fill_temp(row):
        if not np.isnan(row['Temperature']):
            return row['Temperature']
        key = (row['hour'], row['Weather'])
        if key in temp_map and not np.isnan(temp_map[key]):
            return temp_map[key]
        return overall_mean
        
    df['Temperature'] = df.apply(fill_temp, axis=1)
    return df

## Step 3: Feature Engineering Pipeline

In [ ]:
def get_day48_grid(train_df):
    """
    Creates a complete interpolated grid of demand on Day 48 for all geohashes.
    """
    day48 = train_df[train_df['day'] == 48].copy()
    day48 = parse_time_features(day48)
    
    geohashes = day48['geohash'].unique()
    times = sorted(day48['time_minutes'].unique())
    
    # Create complete grid
    grid = pd.MultiIndex.from_product([geohashes, times], names=['geohash', 'time_minutes']).to_frame().reset_index(drop=True)
    grid = grid.merge(day48[['geohash', 'time_minutes', 'demand']], on=['geohash', 'time_minutes'], how='left')
    
    # Sort and interpolate
    grid = grid.sort_values(['geohash', 'time_minutes'])
    grid['demand_interpolated'] = grid.groupby('geohash')['demand'].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both').fillna(0.0)
    )
    
    # Add prefix columns for neighborhood profiling on Day 48
    grid['geohash_prefix5'] = grid['geohash'].str[:5]
    grid['geohash_prefix4'] = grid['geohash'].str[:4]
    
    return grid

def build_features(df, train_df, is_train=True):
    """
    Builds features for df using train_df as context/history.
    """
    # 1. Parse basic time features
    df = parse_time_features(df.copy())
    
    # 2. Decode geohashes
    lats_lons = df['geohash'].apply(decode_geohash)
    df['latitude'] = [x[0] for x in lats_lons]
    df['longitude'] = [x[1] for x in lats_lons]
    
    # 3. Geohash prefixes
    df['geohash_prefix4'] = df['geohash'].str[:4]
    df['geohash_prefix5'] = df['geohash'].str[:5]
    
    # 4. Impute missing values
    if train_df is not None:
        train_parsed = parse_time_features(train_df.copy())
        df = impute_missing_values(df, train_parsed)
    else:
        df = impute_missing_values(df)
        
    # 5. Same-day lag features (lags from 0:00 to 2:00)
    history_0_2 = df[df['time_minutes'] <= 120].copy()
    
    # Pivot to get geohash x day x time_minutes demand
    pivot_0_2 = history_0_2.pivot_table(
        index=['geohash', 'day'], 
        columns='time_minutes', 
        values='demand'
    ).reset_index()
    
    # Ensure all required timestamps are present in pivot columns
    for t in [0, 15, 30, 45, 60, 75, 90, 105, 120]:
        if t not in pivot_0_2.columns:
            pivot_0_2[t] = np.nan
            
    # Interpolate same-day lags row-wise (for each geohash/day)
    lag_cols = [0, 15, 30, 45, 60, 75, 90, 105, 120]
    pivot_0_2[lag_cols] = pivot_0_2[lag_cols].interpolate(axis=1, limit_direction='both').fillna(0.0)
    
    # Compute mean over the 0:00-2:00 window
    pivot_0_2['mean_demand_0_to_2'] = pivot_0_2[lag_cols].mean(axis=1)
    
    # Rename columns for merging
    pivot_0_2 = pivot_0_2.rename(columns={
        120: 'demand_at_2_00',
        105: 'demand_at_1_45',
        90: 'demand_at_1_30',
        75: 'demand_at_1_15',
        60: 'demand_at_1_00'
    })
    
    # Compute neighborhood level same-day lags
    pivot_0_2['geohash_prefix5'] = pivot_0_2['geohash'].str[:5]
    pivot_0_2['geohash_prefix4'] = pivot_0_2['geohash'].str[:4]
    
    prefix5_early = pivot_0_2.groupby(['geohash_prefix5', 'day'])['mean_demand_0_to_2'].mean().reset_index()
    prefix5_early.columns = ['geohash_prefix5', 'day', 'mean_demand_0_to_2_prefix5']
    
    prefix4_early = pivot_0_2.groupby(['geohash_prefix4', 'day'])['mean_demand_0_to_2'].mean().reset_index()
    prefix4_early.columns = ['geohash_prefix4', 'day', 'mean_demand_0_to_2_prefix4']
    
    # Merge same-day lags into df
    df = df.merge(
        pivot_0_2[['geohash', 'day', 'demand_at_2_00', 'demand_at_1_45', 'demand_at_1_30', 'demand_at_1_15', 'demand_at_1_00', 'mean_demand_0_to_2']], 
        on=['geohash', 'day'], 
        how='left'
    )
    df = df.merge(prefix5_early, on=['geohash_prefix5', 'day'], how='left')
    df = df.merge(prefix4_early, on=['geohash_prefix4', 'day'], how='left')
    
    # Fill missing same-day lags with neighborhood values
    df['mean_demand_0_to_2'] = df['mean_demand_0_to_2'].fillna(df['mean_demand_0_to_2_prefix5']).fillna(df['mean_demand_0_to_2_prefix4']).fillna(0.0)
    for col in ['demand_at_2_00', 'demand_at_1_45', 'demand_at_1_30', 'demand_at_1_15', 'demand_at_1_00']:
        df[col] = df[col].fillna(df['mean_demand_0_to_2'])
        
    # 6. Previous day (Day 48) historical features
    if train_df is not None:
        grid48 = get_day48_grid(train_df)
        grid48_clean = grid48[['geohash', 'time_minutes', 'demand_interpolated']].copy()
        
        # We merge Day 48 demand at offset times
        offsets = {
            'demand_day48_t': 0,
            'demand_day48_t_minus_15': -15,
            'demand_day48_t_minus_30': -30,
            'demand_day48_t_minus_60': -60,
            'demand_day48_t_plus_15': 15,
            'demand_day48_t_plus_30': 30,
            'demand_day48_t_plus_60': 60
        }
        
        for col_name, offset in offsets.items():
            temp_time_col = f'temp_time_{offset}'
            df[temp_time_col] = (df['time_minutes'] + offset).clip(0, 1425)
            df[temp_time_col] = (df[temp_time_col] / 15.0).round().astype(int) * 15
            
            df = df.merge(
                grid48_clean.rename(columns={'time_minutes': temp_time_col, 'demand_interpolated': col_name}),
                on=['geohash', temp_time_col],
                how='left'
            )
            df[col_name] = df[col_name].fillna(0.0)
            df = df.drop(columns=[temp_time_col])
        
        # Window rolling statistics on Day 48
        df['demand_day48_rolling_mean_30'] = (df['demand_day48_t_minus_15'] + df['demand_day48_t'] + df['demand_day48_t_plus_15']) / 3.0
        df['demand_day48_rolling_mean_60'] = (df['demand_day48_t_minus_30'] + df['demand_day48_t_minus_15'] + df['demand_day48_t'] + df['demand_day48_t_plus_15'] + df['demand_day48_t_plus_30']) / 5.0
        
        # Day 48 overall geohash statistics
        stats48 = grid48.groupby('geohash')['demand_interpolated'].agg(['mean', 'std', 'max', 'min']).reset_index()
        stats48.columns = ['geohash', 'day48_overall_mean', 'day48_overall_std', 'day48_overall_max', 'day48_overall_min']
        df = df.merge(stats48, on='geohash', how='left')
        
        # Neighborhood stats on Day 48
        prefix5_stats = grid48.groupby('geohash_prefix5')['demand_interpolated'].agg(['mean', 'std', 'max', 'min']).reset_index()
        prefix5_stats.columns = ['geohash_prefix5', 'day48_prefix5_mean', 'day48_prefix5_std', 'day48_prefix5_max', 'day48_prefix5_min']
        df = df.merge(prefix5_stats, on='geohash_prefix5', how='left')
        
        prefix4_stats = grid48.groupby('geohash_prefix4')['demand_interpolated'].agg(['mean', 'std', 'max', 'min']).reset_index()
        prefix4_stats.columns = ['geohash_prefix4', 'day48_prefix4_mean', 'day48_prefix4_std', 'day48_prefix4_max', 'day48_prefix4_min']
        df = df.merge(prefix4_stats, on='geohash_prefix4', how='left')
        
        # Neighborhood profiles on Day 48 at time t
        prefix5_profile = grid48.groupby(['geohash_prefix5', 'time_minutes'])['demand_interpolated'].mean().reset_index()
        prefix5_profile.columns = ['geohash_prefix5', 'time_minutes', 'demand_day48_prefix5_t']
        
        prefix4_profile = grid48.groupby(['geohash_prefix4', 'time_minutes'])['demand_interpolated'].mean().reset_index()
        prefix4_profile.columns = ['geohash_prefix4', 'time_minutes', 'demand_day48_prefix4_t']
        
        df = df.merge(prefix5_profile, on=['geohash_prefix5', 'time_minutes'], how='left')
        df = df.merge(prefix4_profile, on=['geohash_prefix4', 'time_minutes'], how='left')
        
        df['demand_day48_prefix5_t'] = df['demand_day48_prefix5_t'].fillna(0.0)
        df['demand_day48_prefix4_t'] = df['demand_day48_prefix4_t'].fillna(0.0)
        
        # City-wide profile
        city_profile48 = grid48.groupby('time_minutes')['demand_interpolated'].mean().to_dict()
        df['city_profile_day48'] = df['time_minutes'].map(city_profile48)
    else:
        cols_to_nan = [
            'demand_day48_t', 'demand_day48_t_minus_15', 'demand_day48_t_minus_30', 'demand_day48_t_minus_60', 
            'demand_day48_t_plus_15', 'demand_day48_t_plus_30', 'demand_day48_t_plus_60', 
            'demand_day48_rolling_mean_30', 'demand_day48_rolling_mean_60', 
            'day48_overall_mean', 'day48_overall_std', 'day48_overall_max', 'day48_overall_min',
            'day48_prefix5_mean', 'day48_prefix5_std', 'day48_prefix5_max', 'day48_prefix5_min',
            'day48_prefix4_mean', 'day48_prefix4_std', 'day48_prefix4_max', 'day48_prefix4_min',
            'demand_day48_prefix5_t', 'demand_day48_prefix4_t', 'city_profile_day48'
        ]
        for col in cols_to_nan:
            df[col] = 0.0
            
    # Final NaNs safety check for features only (not target)
    feature_cols = [c for c in df.columns if c != 'demand']
    df[feature_cols] = df[feature_cols].fillna(0.0)
    
    return df

## Step 4: Loading Data and Training Pipeline

In [ ]:
print("Loading datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print("Building features...")
all_df = pd.concat([train, test], ignore_index=True)
all_feat = build_features(all_df, train)

train_feat = all_feat[all_feat['demand'].notna()].copy()
test_feat = all_feat[all_feat['demand'].isna()].copy()

cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'geohash', 'geohash_prefix4', 'geohash_prefix5']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

train_feat[cat_cols] = encoder.fit_transform(train_feat[cat_cols].astype(str))
test_feat[cat_cols] = encoder.transform(test_feat[cat_cols].astype(str))

features = [
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather',
    'hour', 'minute', 'time_minutes', 'sin_time', 'cos_time', 'latitude', 'longitude',
    'geohash', 'geohash_prefix4', 'geohash_prefix5',
    'demand_at_2_00', 'demand_at_1_45', 'demand_at_1_30', 'demand_at_1_15', 'demand_at_1_00', 'mean_demand_0_to_2',
    'mean_demand_0_to_2_prefix5', 'mean_demand_0_to_2_prefix4',
    'demand_day48_t', 'demand_day48_t_minus_15', 'demand_day48_t_minus_30', 'demand_day48_t_minus_60',
    'demand_day48_t_plus_15', 'demand_day48_t_plus_30', 'demand_day48_t_plus_60',
    'demand_day48_rolling_mean_30', 'demand_day48_rolling_mean_60',
    'day48_overall_mean', 'day48_overall_std', 'day48_overall_max', 'day48_overall_min',
    'day48_prefix5_mean', 'day48_prefix5_std', 'day48_prefix5_max', 'day48_prefix5_min',
    'day48_prefix4_mean', 'day48_prefix4_std', 'day48_prefix4_max', 'day48_prefix4_min',
    'demand_day48_prefix5_t', 'demand_day48_prefix4_t', 'city_profile_day48'
]

target = 'demand'

train_48 = train_feat[train_feat['day'] == 48].copy()
train_49 = train_feat[train_feat['day'] == 49].copy()

## Step 5: Training and Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(train_49))
oof_xgb = np.zeros(len(train_49))
oof_cb = np.zeros(len(train_49))

test_lgb = np.zeros(len(test_feat))
test_xgb = np.zeros(len(test_feat))
test_cb = np.zeros(len(test_feat))

lgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

xgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': 0
}

cb_params = {
    'iterations': 1500,
    'learning_rate': 0.03,
    'depth': 6,
    'random_seed': 42,
    'verbose': 0
}

for fold, (train_idx, val_idx) in enumerate(kf.split(train_49)):
    print(f"\n--- Training Fold {fold} ---")
    fold_train_49 = train_49.iloc[train_idx]
    fold_train = pd.concat([train_48, fold_train_49], ignore_index=True)
    
    X_train, y_train = fold_train[features], fold_train[target]
    fold_val = train_49.iloc[val_idx]
    X_val, y_val = fold_val[features], fold_val[target]
    
    # 1. LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = np.clip(model_lgb.predict(X_val), 0.0, 1.0)
    test_lgb += np.clip(model_lgb.predict(test_feat[features]), 0.0, 1.0) / 5.0
    
    # 2. XGBoost
    model_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = np.clip(model_xgb.predict(X_val), 0.0, 1.0)
    test_xgb += np.clip(model_xgb.predict(test_feat[features]), 0.0, 1.0) / 5.0
    
    # 3. CatBoost
    model_cb = cb.CatBoostRegressor(**cb_params, early_stopping_rounds=50)
    model_cb.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
    oof_cb[val_idx] = np.clip(model_cb.predict(X_val), 0.0, 1.0)
    test_cb += np.clip(model_cb.predict(test_feat[features]), 0.0, 1.0) / 5.0
    
    r2_lgb = r2_score(y_val, oof_lgb[val_idx]) * 100
    r2_xgb = r2_score(y_val, oof_xgb[val_idx]) * 100
    r2_cb = r2_score(y_val, oof_cb[val_idx]) * 100
    print(f"Fold {fold} R2 Scores -> LGBM: {r2_lgb:.4f} | XGB: {r2_xgb:.4f} | CatBoost: {r2_cb:.4f}")

## Step 6: Out-of-Fold Evaluation

In [ ]:
y_49 = train_49[target].values
score_lgb = r2_score(y_49, oof_lgb) * 100
score_xgb = r2_score(y_49, oof_xgb) * 100
score_cb = r2_score(y_49, oof_cb) * 100
oof_ensemble = (oof_lgb + oof_xgb + oof_cb) / 3.0
score_ensemble = r2_score(y_49, oof_ensemble) * 100

print("="*50)
print(f"OOF R2 Score LightGBM: {score_lgb:.4f}")
print(f"OOF R2 Score XGBoost:  {score_xgb:.4f}")
print(f"OOF R2 Score CatBoost: {score_cb:.4f}")
print(f"OOF R2 Score Ensemble: {score_ensemble:.4f}")
print("="*50)

## Step 7: Save and Verify Submission

In [ ]:
final_preds = (test_lgb + test_xgb + test_cb) / 3.0
submission = pd.DataFrame({
    'Index': test_feat['Index'].astype(int),
    'demand': np.clip(final_preds, 0.0, 1.0)
})

# Verification asserts
assert len(submission) == 41778, f"Incorrect number of rows: {len(submission)}"
assert list(submission.columns) == ['Index', 'demand'], f"Incorrect columns: {list(submission.columns)}"
assert not submission['demand'].isna().any(), "Submission contains NaN values"
assert (submission['demand'] >= 0.0).all() and (submission['demand'] <= 1.0).all(), "Predictions out of bounds"

test_orig = pd.read_csv('test.csv')
assert (submission['Index'].values == test_orig['Index'].values).all(), "Index values do not match test file"

submission.to_csv('submission.csv', index=False)
print("Submission successfully saved and verified!")